In [ ]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [ ]:
anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

In [ ]:
from member import Member
from conversation_state import ConversationState
from conversation_context import ConversationContext
from conversation_role import ConversationRole
import random

# Setup the board
conversation_state = ConversationState.OPEN
conversation_context = ConversationContext(ConversationState.OPEN)
Member.set_shared_context(conversation_context)

board = [
    Member("Anna Bellini", anthropic_url, anthropic_api_key, "claude-sonnet-4-5-20250929", "Chairman"),
    Member("Giorgio Pagani", gemini_url, google_api_key, "gemini-2.5-pro", "CEO, Board member"),
    Member("Wang Lei Choo", deepseek_url, deepseek_api_key, "deepseek-reasoner", "CTO, Board member"),
    Member("Ryan O'Donoghue", groq_url, groq_api_key, "openai/gpt-oss-120b", "VP Marketing, board member"),
    Member("John Rust", grok_url, grok_api_key, "grok-4", "Board member, AI Adviser"),
    Member("Olga Klenova", openrouter_url, openrouter_api_key, "z-ai/glm-4.5", "Board member, HR Adviser")
]

board[0].set_conversation_role(ConversationRole.CHAIRMAN)
board[len(board)-1].set_conversation_role(ConversationRole.SECRETARY)

experts = random.sample(range(1, 5), 2)
print(f"Company Board:")
for index, member in enumerate(board):
    if index in experts:
        member.set_conversation_role(ConversationRole.EXPERT)
    elif member.conversation_role == ConversationRole.NONE:
        member.set_conversation_role(ConversationRole.AUDITOR)
    print(f"\t{member.name} is {member.conversation_role.value}")


In [ ]:
# the board meeting
conversation_context.reset()
print("Starting the board meeting...")
subject = "Our company latest P&L shows sharp decline in revenue and we will not enough cash to continue operation in the next quarter if we dont find a solution."
conversation_context.subject = subject
print(f"\nSubject: {subject}")

def print_markdown(text):
    display(Markdown(text))

conversation_context.add_callback(ConversationState.QUESTION, print_markdown)
conversation_context.add_callback(ConversationState.DECISION, print_markdown)
conversation_context.add_callback(ConversationState.SUMMARY, print_markdown)

while True:
    conversation_state = conversation_context.get_conversation_state()
    print(f"Current conversation state: {conversation_state.value}")
    for member in board:
        conversation_role = member.conversation_role
        if conversation_context.should_participate(conversation_role):
            print(f"\t{member.name}")
            response = member.get_member_response()
            conversation_context.add_response(response)
    conversation_context.update_context()

    if conversation_state == ConversationState.CLOSE:
        break
conversation_context.print_context()
